In [ ]:
'application_train.csv'

'application_train.csv'

# Modelisation

In [11]:
import numpy as np
import pandas as pd
import gc
import time
from contextlib import contextmanager
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import KFold, StratifiedKFold
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

ModuleNotFoundError: No module named 'lightgbm'

In [6]:
pip install pandas

   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------- ----------------------- 4.5/11.0 MB 26.9 MB/s eta 0:00:01
   ---------------------------------------- 11.0/11.0 MB 31.3 MB/s  0:00:00

   ---------------------------------------- 0/2 [pytz]
   ---------------------------------------- 0/2 [pytz]
   ---------------------------------------- 0/2 [pytz]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ----------------

In [2]:
pd.set_option('display.max_row', 20)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 500)

In [12]:
@contextmanager
def timer(title):
    t0 = time.time()
    yield
    print("{} - done in {:.0f}s".format(title, time.time() - t0))

# Import des bases de données

In [13]:
df = pd.read_csv('data.csv', nrows=None)
df.shape

(356251, 463)

In [14]:
# File system management
import os
import pickle
import dill

from collections import Counter

# Data manipulation
import numpy as np
import pandas as pd

# Scikit-learn
from sklearn.model_selection import cross_val_predict, cross_validate, cross_val_score, train_test_split
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_curve, roc_auc_score, f1_score, accuracy_score, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import make_scorer
from sklearn.preprocessing import StandardScaler

# Hyperparameters optimisation
from hyperopt import tpe, hp, fmin, space_eval, STATUS_OK, Trials, SparkTrials
from hyperopt.pyll.base import scope

# XGBoost
import xgboost as xgb

# LightGBM
import lightgbm as ltb

# Balancing data
from imblearn.over_sampling import SMOTE

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Interpratability

import shap


# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(palette="Set1")

ModuleNotFoundError: No module named 'dill'

In [15]:
df = df.replace([np.inf, -np.inf], np.nan)

# Split du dataset

In [16]:
df.shape

(356251, 463)

In [17]:
# Define usefull variables
TARGET = 'TARGET'
ID = 'SK_ID_CURR'

In [18]:
# === Split train / test avant tout ===
train_df = df[df['TARGET'].notna()].copy()
test_df  = df[df['TARGET'].isna()].copy()


In [19]:

pip install dill

Note: you may need to restart the kernel to use updated packages.


In [ ]:
!pip install pyarrow --quiet

In [20]:
display(test_df.head())
display(train_df.head())

,Unnamed: 0,SK_ID_CURR,TARGET,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,INSTAL_AMT_INSTALMENT_MEAN,INSTAL_AMT_INSTALMENT_SUM,INSTAL_AMT_PAYMENT_MIN,INSTAL_AMT_PAYMENT_MAX,INSTAL_AMT_PAYMENT_MEAN,INSTAL_AMT_PAYMENT_SUM,INSTAL_DAYS_ENTRY_PAYMENT_MAX,INSTAL_DAYS_ENTRY_PAYMENT_MEAN,INSTAL_DAYS_ENTRY_PAYMENT_SUM,INSTAL_COUNT
307507,307507,100001,NaN,1,0,0,0,135000.0,568800.0,20560.5,...,5885.1323,41195.926,3951.000,17397.900,5885.1323,41195.926,-1628.0,-2195.00000,-15365.0,7.0
307508,307508,100005,NaN,0,0,0,0,99000.0,222768.0,17370.0,...,6240.2050,56161.844,4813.200,17656.244,6240.2050,56161.844,-470.0,-609.55554,-5486.0,9.0
307509,307509,100013,NaN,0,1,0,0,202500.0,663264.0,69777.0,...,10897.8980,1689174.200,6.165,357347.750,9740.2350,1509736.500,-14.0,-1358.10960,-210507.0,155.0
307510,307510,100028,NaN,1,0,0,2,315000.0,1575000.0,49018.5,...,4979.2820,562658.900,1.170,38988.540,4356.7314,492310.660,-29.0,-858.54865,-97016.0,113.0
307511,307511,100038,NaN,0,1,1,1,180000.0,625500.0,32067.0,...,11100.3380,133204.050,11097.450,11100.600,11100.3380,133204.050,-466.0,-634.25000,-7611.0,12.0


,Unnamed: 0,SK_ID_CURR,TARGET,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,INSTAL_AMT_INSTALMENT_MEAN,INSTAL_AMT_INSTALMENT_SUM,INSTAL_AMT_PAYMENT_MIN,INSTAL_AMT_PAYMENT_MAX,INSTAL_AMT_PAYMENT_MEAN,INSTAL_AMT_PAYMENT_SUM,INSTAL_DAYS_ENTRY_PAYMENT_MAX,INSTAL_DAYS_ENTRY_PAYMENT_MEAN,INSTAL_DAYS_ENTRY_PAYMENT_SUM,INSTAL_COUNT
0,0,100002,1.0,0,0,0,0,202500.0,406597.5,24700.5,...,11559.247,219625.690,9251.775,53093.746,11559.247,219625.690,-49.0,-315.42105,-5993.0,19.0
1,1,100003,0.0,1,0,1,0,270000.0,1293502.5,35698.5,...,64754.586,1618864.600,6662.970,560835.400,64754.586,1618864.600,-544.0,-1385.32000,-34633.0,25.0
2,2,100004,0.0,0,1,0,0,67500.0,135000.0,6750.0,...,7096.155,21288.465,5357.250,10573.965,7096.155,21288.465,-727.0,-761.66670,-2285.0,3.0
3,3,100006,0.0,1,0,0,0,135000.0,312682.5,29686.5,...,62947.090,1007153.440,2482.920,691786.900,62947.090,1007153.440,-12.0,-271.62500,-4346.0,16.0
4,4,100007,0.0,0,0,0,0,121500.0,513000.0,21865.5,...,12666.444,835985.300,0.180,22678.785,12214.061,806128.000,-14.0,-1032.24240,-68128.0,66.0


In [21]:
# variables
TARGET = 'TARGET'
ID = 'SK_ID_CURR'

In [22]:
test_df = test_df.drop(columns=['Unnamed: 0'])
train_df= train_df.drop(columns=['Unnamed: 0'])

In [23]:
display(test_df.shape)
display(train_df.shape)

display(test_df.head())
display(train_df.head())

(48744, 462)

(307507, 462)

,SK_ID_CURR,TARGET,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,REGION_POPULATION_RELATIVE,...,INSTAL_AMT_INSTALMENT_MEAN,INSTAL_AMT_INSTALMENT_SUM,INSTAL_AMT_PAYMENT_MIN,INSTAL_AMT_PAYMENT_MAX,INSTAL_AMT_PAYMENT_MEAN,INSTAL_AMT_PAYMENT_SUM,INSTAL_DAYS_ENTRY_PAYMENT_MAX,INSTAL_DAYS_ENTRY_PAYMENT_MEAN,INSTAL_DAYS_ENTRY_PAYMENT_SUM,INSTAL_COUNT
307507,100001,NaN,1,0,0,0,135000.0,568800.0,20560.5,0.018850,...,5885.1323,41195.926,3951.000,17397.900,5885.1323,41195.926,-1628.0,-2195.00000,-15365.0,7.0
307508,100005,NaN,0,0,0,0,99000.0,222768.0,17370.0,0.035792,...,6240.2050,56161.844,4813.200,17656.244,6240.2050,56161.844,-470.0,-609.55554,-5486.0,9.0
307509,100013,NaN,0,1,0,0,202500.0,663264.0,69777.0,0.019101,...,10897.8980,1689174.200,6.165,357347.750,9740.2350,1509736.500,-14.0,-1358.10960,-210507.0,155.0
307510,100028,NaN,1,0,0,2,315000.0,1575000.0,49018.5,0.026392,...,4979.2820,562658.900,1.170,38988.540,4356.7314,492310.660,-29.0,-858.54865,-97016.0,113.0
307511,100038,NaN,0,1,1,1,180000.0,625500.0,32067.0,0.010032,...,11100.3380,133204.050,11097.450,11100.600,11100.3380,133204.050,-466.0,-634.25000,-7611.0,12.0


,SK_ID_CURR,TARGET,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,REGION_POPULATION_RELATIVE,...,INSTAL_AMT_INSTALMENT_MEAN,INSTAL_AMT_INSTALMENT_SUM,INSTAL_AMT_PAYMENT_MIN,INSTAL_AMT_PAYMENT_MAX,INSTAL_AMT_PAYMENT_MEAN,INSTAL_AMT_PAYMENT_SUM,INSTAL_DAYS_ENTRY_PAYMENT_MAX,INSTAL_DAYS_ENTRY_PAYMENT_MEAN,INSTAL_DAYS_ENTRY_PAYMENT_SUM,INSTAL_COUNT
0,100002,1.0,0,0,0,0,202500.0,406597.5,24700.5,0.018801,...,11559.247,219625.690,9251.775,53093.746,11559.247,219625.690,-49.0,-315.42105,-5993.0,19.0
1,100003,0.0,1,0,1,0,270000.0,1293502.5,35698.5,0.003541,...,64754.586,1618864.600,6662.970,560835.400,64754.586,1618864.600,-544.0,-1385.32000,-34633.0,25.0
2,100004,0.0,0,1,0,0,67500.0,135000.0,6750.0,0.010032,...,7096.155,21288.465,5357.250,10573.965,7096.155,21288.465,-727.0,-761.66670,-2285.0,3.0
3,100006,0.0,1,0,0,0,135000.0,312682.5,29686.5,0.008019,...,62947.090,1007153.440,2482.920,691786.900,62947.090,1007153.440,-12.0,-271.62500,-4346.0,16.0
4,100007,0.0,0,0,0,0,121500.0,513000.0,21865.5,0.028663,...,12666.444,835985.300,0.180,22678.785,12214.061,806128.000,-14.0,-1032.24240,-68128.0,66.0


# Split feature target

In [24]:
X = train_df.drop(columns=[ID, TARGET])
y = train_df[TARGET].astype(int)

X_test = test_df.drop(columns=[ID], errors="ignore")

**Split interne**

In [25]:
# Étape 1 : Split interne
from sklearn.model_selection import train_test_split

X_tr, X_va, y_tr, y_va = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [26]:
# Étape 1 : Split interne
from sklearn.model_selection import train_test_split

X_tr, X_va, y_tr, y_va = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Étape 2 : Modèle baseline sans équilibrage
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=2000,
        solver="lbfgs",
        random_state=42
    ))
])

# Étape 3 : Entraînement et prédictions
baseline_model.fit(X_tr, y_tr)

# prédictions
proba_va = baseline_model.predict_proba(X_va)[:, 1]  # probas classe 1
y_pred_05 = (proba_va >= 0.5).astype(int)            # seuil 0.5

# Étape 4 : Évaluation
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    average_precision_score, classification_report,
    confusion_matrix
)

print("ROC-AUC :", roc_auc_score(y_va, proba_va))
print("PR-AUC  :", average_precision_score(y_va, proba_va))
print("Accuracy:", accuracy_score(y_va, y_pred_05))
print("F1      :", f1_score(y_va, y_pred_05))
print("\n=== Classification report (0.5) ===")
print(classification_report(y_va, y_pred_05, digits=3))
print("Confusion matrix (0.5):\n", confusion_matrix(y_va, y_pred_05))


ROC-AUC : 0.7687201962635632
PR-AUC  : 0.2491929441730797
Accuracy: 0.9192221391174271
F1      : 0.047180667433831994

=== Classification report (0.5) ===
              precision    recall  f1-score   support

           0      0.921     0.998     0.958     56537
           1      0.494     0.025     0.047      4965

    accuracy                          0.919     61502
   macro avg      0.707     0.511     0.503     61502
weighted avg      0.886     0.919     0.884     61502

Confusion matrix (0.5):
 [[56411   126]
 [ 4842   123]]



ROC-AUC	0.77 : Assez correct — le modèle sépare un peu les bons/mauvais clients, mais ce n’est pas encore suffisant.

ROC-AUC	0.77 : Assez correct — le modèle sépare un peu les bons/mauvais clients, mais ce n’est pas encore suffisant.
PR-AUC	0.25 : Faible — le modèle ne repère pas bien les mauvais payeurs (classe 1), car le jeu est très déséquilibré.
Accuracy	0.92	Trompeuse ici, car 92 % des gens ne font pas défaut → le modèle prédit presque toujours “0”.
F1	0.048	Très faible → confirme que la classe 1 est très mal détectée.

#Baseline (Sans équilibrage de données)

In [15]:
import numpy as np, gc
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score, classification_report, confusion_matrix

# Pipeline NUMÉRIQUE SEUL (mémoire light)
baseline_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("to32", FunctionTransformer(lambda X: X.astype(np.float32), validate=False)),
    ("scaler", StandardScaler(with_mean=False)),  # évite densification
    ("clf", LogisticRegression(
        solver="saga",
        penalty="l2",
        C=1.0,
        max_iter=3000,
        n_jobs=-1,
        random_state=42
    ))
])

# Fit
baseline_model.fit(X_tr, y_tr)
# Libère l'entraînement avant la validation pour gagner de la RAM
del X_tr, y_tr; gc.collect()

# Predict proba en validation
proba_va = baseline_model.predict_proba(X_va)[:, 1]
y_pred_05 = (proba_va >= 0.5).astype(int)

print("ROC-AUC :", roc_auc_score(y_va, proba_va))
print("PR-AUC  :", average_precision_score(y_va, proba_va))
print("Accuracy:", accuracy_score(y_va, y_pred_05))
print("F1      :", f1_score(y_va, y_pred_05))
print("\n=== Classification report (0.5) ===")
print(classification_report(y_va, y_pred_05, digits=3))
print("Confusion matrix (0.5):\n", confusion_matrix(y_va, y_pred_05))


ROC-AUC : 0.7670606960754573
PR-AUC  : 0.24642445058798262
Accuracy: 0.9192871776527592
F1      : 0.03160358954350371

=== Classification report (0.5) ===
              precision    recall  f1-score   support

           0      0.920     0.999     0.958     56537
           1      0.503     0.016     0.032      4965

    accuracy                          0.919     61502
   macro avg      0.712     0.507     0.495     61502
weighted avg      0.887     0.919     0.883     61502

Confusion matrix (0.5):
 [[56457    80]
 [ 4884    81]]


In [ ]:
import numpy as np, gc
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score, classification_report, confusion_matrix

# Pipeline NUMÉRIQUE SEUL (mémoire light)
baseline_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("to32", FunctionTransformer(lambda X: X.astype(np.float32), validate=False)),
    ("scaler", StandardScaler(with_mean=False)),  # évite densification
    ("clf", LogisticRegression(
        solver="saga",
        penalty="l2",
        C=1.0,
        max_iter=3000,
        n_jobs=-1,
        random_state=42
    ))
])

# Fit
baseline_model.fit(X_tr, y_tr)
# Libère l'entraînement avant la validation pour gagner de la RAM
del X_tr, y_tr; gc.collect()

# Predict proba en validation
proba_va = baseline_model.predict_proba(X_va)[:, 1]
y_pred_05 = (proba_va >= 0.5).astype(int)

print("ROC-AUC :", roc_auc_score(y_va, proba_va))
print("PR-AUC  :", average_precision_score(y_va, proba_va))
print("Accuracy:", accuracy_score(y_va, y_pred_05))
print("F1      :", f1_score(y_va, y_pred_05))
print("\n=== Classification report (0.5) ===")
print(classification_report(y_va, y_pred_05, digits=3))
print("Confusion matrix (0.5):\n", confusion_matrix(y_va, y_pred_05))

ROC-AUC : 0.7670680489588749
PR-AUC  : 0.24636320891688945
Accuracy: 0.9193034372865923
F1      : 0.03198751706651063

=== Classification report (0.5) ===
              precision    recall  f1-score   support

           0      0.920     0.999     0.958     56537
           1      0.506     0.017     0.032      4965

    accuracy                          0.919     61502
   macro avg      0.713     0.508     0.495     61502
weighted avg      0.887     0.919     0.883     61502

Confusion matrix (0.5):
 [[56457    80]
 [ 4883    82]]


#Equilibrage de données

In [27]:
pip install -U imbalanced-learn scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [28]:

# pip install imbalanced-learn

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    make_scorer, accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.model_selection import StratifiedKFold, cross_validate

from imblearn.pipeline import Pipeline as ImbPipeline


In [29]:

import gc
import numpy as np
import pandas as pd

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    make_scorer, accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.model_selection import StratifiedKFold, cross_validate

# Paramètres de base pour la logreg
logreg_params = {"max_iter": 2000, "solver": "lbfgs", "random_state": 42}

# CV légère pour RAM
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Métriques
scoring = {
    "roc_auc":   "roc_auc",
    "pr_auc":    "average_precision",
    "accuracy":  make_scorer(accuracy_score),
    "precision": make_scorer(precision_score, average="binary", zero_division=0),
    "recall":    make_scorer(recall_score, average="binary", zero_division=0),
    "f1":        make_scorer(f1_score, average="binary", zero_division=0),
}

def cv_table(model, X, y):
    """Retourne un DataFrame (moyenne et écart-type) pour chaque métrique."""
    out = cross_validate(
        model, X, y, cv=cv, scoring=scoring,
        n_jobs=1,              # éviter pics RAM
        error_score="raise",
        return_train_score=False
    )
    rows = []
    for k, v in out.items():
        if k.startswith("test_"):
            metric = k.replace("test_", "")
            rows.append({"metric": metric,
                         "mean": float(np.mean(v)),
                         "std":  float(np.std(v))})
    rep = pd.DataFrame(rows).set_index("metric").sort_index()
    return rep


In [30]:
logreg_params = {"max_iter": 2000, "solver": "lbfgs", "random_state": 42}


1) LogReg avec class_weight="balanced"

In [31]:
pipe_weight_logreg = ImbPipeline(steps=[
    ("imp",   SimpleImputer(strategy="median")),
    ("scal",  StandardScaler(with_mean=True)),
    ("clf",   LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])

rep_weight = cv_table(pipe_weight_logreg, X, y)
print("\n=== LogReg_weight ===")
print(rep_weight.sort_values("mean", ascending=False))




=== LogReg_weight ===
               mean       std
metric                       
roc_auc    0.768256  0.001833
accuracy   0.702446  0.001086
recall     0.700302  0.003372
f1         0.275362  0.000624
pr_auc     0.241480  0.001681
precision  0.171374  0.000384


In [ ]:
del pipe_over_logreg; gc.collect()
del pipe_under_logreg; gc.collect()
del pipe_smote_under_logreg; gc.collect()
del pipe_smote_rf; gc.collect()


2) LogReg + SMOTE (oversampling)

In [32]:
pipe_over_logreg = ImbPipeline(steps=[
    ("imp",   SimpleImputer(strategy="median")),
    ("scal",  StandardScaler(with_mean=True)),
    ("over",  SMOTE(sampling_strategy=0.5, random_state=42)),
    ("clf",   LogisticRegression(**logreg_params))
])

rep_over = cv_table(pipe_over_logreg, X, y)
print("\n=== LogReg_SMOTE ===")
print(rep_over.sort_values("mean", ascending=False))



=== LogReg_SMOTE ===
               mean       std
metric                       
accuracy   0.832384  0.000486
roc_auc    0.753808  0.002319
recall     0.443182  0.000827
f1         0.299184  0.000611
pr_auc     0.230015  0.000814
precision  0.225814  0.000659


3) LogReg + undersampling

In [33]:
pipe_under_logreg = ImbPipeline(steps=[
    ("imp",   SimpleImputer(strategy="median")),
    ("scal",  StandardScaler(with_mean=True)),
    ("under", RandomUnderSampler(sampling_strategy=0.5, random_state=42)),
    ("clf",   LogisticRegression(**logreg_params))
])

rep_under = cv_table(pipe_under_logreg, X, y)
print("\n=== LogReg_Under ===")
print(rep_under.sort_values("mean", ascending=False))



=== LogReg_Under ===
               mean       std
metric                       
accuracy   0.840726  0.001123
roc_auc    0.766828  0.001919
recall     0.446002  0.004553
f1         0.311358  0.003556
precision  0.239159  0.002941
pr_auc     0.238894  0.000609


LogReg + SMOTE puis undersampling

In [34]:
pipe_smote_under_logreg = ImbPipeline(steps=[
    ("imp",   SimpleImputer(strategy="median")),
    ("scal",  StandardScaler(with_mean=True)),
    ("over",  SMOTE(sampling_strategy=0.5, random_state=42)),
    ("under", RandomUnderSampler(sampling_strategy=1.0, random_state=42)),
    ("clf",   LogisticRegression(max_iter=2000, random_state=42))
])

rep_smote_under = cv_table(pipe_smote_under_logreg, X, y)
print("\n=== LogReg_SMOTE_Under ===")
print(rep_smote_under.sort_values("mean", ascending=False))




=== LogReg_SMOTE_Under ===
               mean       std
metric                       
roc_auc    0.754549  0.002022
accuracy   0.709912  0.001138
recall     0.663323  0.004974
f1         0.269648  0.002248
pr_auc     0.229199  0.000601
precision  0.169218  0.001447


In [35]:
# comparaison
comparatif = []
if 'rep_weight' in globals():      comparatif.append(("LogReg_weight", rep_weight))
if 'rep_over' in globals():        comparatif.append(("LogReg_SMOTE", rep_over))
if 'rep_under' in globals():       comparatif.append(("LogReg_Under", rep_under))
if 'rep_smote_under' in globals(): comparatif.append(("LogReg_SMOTE_Under", rep_smote_under))


if comparatif:
    table = pd.concat(
        {name: rep['mean'] for name, rep in comparatif},
        axis=1
    )
    print("\n=== Comparatif (moyennes CV) ===")
    display(table.sort_values(by=table.columns[0], ascending=False))


=== Comparatif (moyennes CV) ===


,LogReg_weight,LogReg_SMOTE,LogReg_Under,LogReg_SMOTE_Under
metric,,,,
roc_auc,0.768256,0.753808,0.766828,0.754549
accuracy,0.702446,0.832384,0.840726,0.709912
recall,0.700302,0.443182,0.446002,0.663323
f1,0.275362,0.299184,0.311358,0.269648
pr_auc,0.241480,0.230015,0.238894,0.229199
precision,0.171374,0.225814,0.239159,0.169218


In [ ]:
#del pipe_over_logreg; gc.collect()
#del pipe_under_logreg; gc.collect()
#del pipe_smote_under_logreg; gc.collect()
#del pipe_smote_rf; gc.collect()

In [36]:
# Sélection du meilleur pipeline selon PR-AUC
reports = {
    "LogReg_weight": cv_table(pipe_weight_logreg, X, y),
    "LogReg_SMOTE": cv_table(pipe_over_logreg, X, y),
    "LogReg_Under": cv_table(pipe_under_logreg, X, y),
    "LogReg_SMOTE_Under": cv_table(pipe_smote_under_logreg, X, y),
}

def model_key(name):
    rep = reports[name]
    return rep.loc["pr_auc", "mean"]

best_name = max(reports.keys(), key=model_key)
best_model = {
    "LogReg_weight": pipe_weight_logreg,
    "LogReg_SMOTE": pipe_over_logreg,
    "LogReg_Under": pipe_under_logreg,
    "LogReg_SMOTE_Under": pipe_smote_under_logreg,
}[best_name]

print(">>> Modèle choisi :", best_name)


>>> Modèle choisi : LogReg_weight


# Optimisation métier

objectif métier = détecter un maximum de défauts sans perdre trop de précision.

Quand le modèle prédit si un client remboursera (0) ou fera défaut (1), il peut se tromper de deux façons :

| Type d’erreur         | Cas réel           | Prédiction du modèle | Signification                                   |
| --------------------- | ------------------ | -------------------- | ----------------------------------------------- |
| **Faux positif (FP)** | Bon client (0)     | Mauvais client (1)   | on rejette un bon client à tort                |
| **Faux négatif (FN)** | Mauvais client (1) | Bon client (0)       | on accepte un client qui ne remboursera pas |

Le coût métier de ces erreurs

- Si on refuse un bon client (FP) → on perd un peu de chiffre d’affaires

- Si on accepte un mauvais client (FN) → on perd beaucoup d’argent car il ne rembourse pas

Un faux négatif coûte plus cher qu’un faux positif.

On préfère repérer le plus possible de mauvais payeurs,
même si cela veut dire refuser à tort quelques bons clients.


1 faux négatif (FN) coûte 10× plus cher qu’un faux positif (FP).
En d’autres termes, on valorise la détection des défauts, quitte à avoir un peu plus de faux positifs.


In [ ]:
objectif métier = détecter un maximum de défauts sans perdre trop de précision

In [ ]:
THRESHOLDS = np.linspace(0.01, 0.99, 99)
best_t, best_cost = None, np.inf

for t in THRESHOLDS:
    y_pred = (proba_va >= t).astype(int)
    cost = business_cost(y_va, y_pred, w_fn=0.9, w_fp=0.1)
    if cost < best_cost:
        best_cost, best_t = cost, t

print(f"Seuil métier optimal = {best_t:.3f} | Coût moyen minimal = {best_cost:.4f}")


In [37]:
# Après benchmark multi-pipelines
best_model.fit(X_tr, y_tr)
proba_va = best_model.predict_proba(X_va)[:, 1]

COST_FN, COST_FP = 0.9, 0.1
THRESHOLDS = np.linspace(0.01, 0.99, 99)

def business_cost(y_true, y_pred, cost_fn, cost_fp):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return fn*cost_fn + fp*cost_fp

best_t, best_cost = None, np.inf
for t in THRESHOLDS:
    y_pred = (proba_va >= t).astype(int)
    cost = business_cost(y_va, y_pred, COST_FN, COST_FP)
    if cost < best_cost:
        best_cost, best_t = cost, t

print(f"Seuil métier optimal = {best_t:.3f} | Coût minimal = {best_cost:.0f}")


Seuil métier optimal = 0.540 | Coût minimal = 2954


| Élément                           | Signification                                                                                                              |
| --------------------------------- | -------------------------------------------------------------------------------------------------------------------------- |
| **Seuil = 0.540**                 | le modèle doit classer un client comme “défaillant” s’il a **plus de 54 %** de probabilité estimée.                       |
| **Coût minimal = 2966**           | En appliquant ce seuil, on minimise le **coût métier global** (faux négatifs + faux positifs pondérés).                  |
| **COST_FN / COST_FP = 0.9 / 0.1** | on a décidé qu’un **faux négatif (défaut non détecté)** coûte 9× plus cher qu’un **faux positif (client refusé à tort)**. |


En d’autres termes

Si tu mets le seuil plus bas (< 0.54) → tu détectes plus de mauvais clients,
mais tu refuses plus de bons → ton coût augmente à cause des faux positifs.

Si tu mets le seuil plus haut (> 0.54) → tu acceptes plus de mauvais clients,
donc ton coût augmente à cause des faux négatifs.

Le 0.54 équilibre ces deux effets selon tes poids métier (0.9 / 0.1).

In [38]:
y_pred_opt = (proba_va >= 0.54).astype(int)

print("Confusion matrix (seuil optimal):")
print(confusion_matrix(y_va, y_pred_opt))

print("\nClassification report:")
print(classification_report(y_va, y_pred_opt, digits=3))


Confusion matrix (seuil optimal):
[[42512 14025]
 [ 1724  3241]]

Classification report:
              precision    recall  f1-score   support

           0      0.961     0.752     0.844     56537
           1      0.188     0.653     0.292      4965

    accuracy                          0.744     61502
   macro avg      0.574     0.702     0.568     61502
weighted avg      0.899     0.744     0.799     61502



In [ ]:
après application du seuil métier optimal (≈ 0.54 ), on détecte 65 % des défauts (recall ) tout en gardant une précision correcte (≈ 19 %).


Étape 1 — Confirmer sur la Cross-Validation (CV)
le seuil est trouvé sur le split interne (validation), mais il faut s’assurer qu’il généralise bien :

In [39]:
# valider le même seuil sur plusieurs folds
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scores = []

for tr_idx, va_idx in cv.split(X, y):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    best_model.fit(X_tr, y_tr)
    proba_va = best_model.predict_proba(X_va)[:, 1]
    y_pred_opt = (proba_va >= 0.54).astype(int)

    auc = roc_auc_score(y_va, proba_va)
    pr  = average_precision_score(y_va, proba_va)
    scores.append((auc, pr))

print("ROC-AUC moyen:", np.mean([s[0] for s in scores]))
print("PR-AUC moyen :", np.mean([s[1] for s in scores]))


ROC-AUC moyen: 0.7682561501667576
PR-AUC moyen : 0.2414798897558091


Étape 2 — Appliquer le modèle final sur tout le jeu d’entraînement

In [40]:
best_model.fit(X, y)

,steps,"[('imp', ...), ('scal', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


Étape 3 — Prédire sur le jeu de test final

In [41]:
X_test = X_test.drop(columns=["TARGET"], errors="ignore")

In [42]:
proba_test = best_model.predict_proba(X_test)[:, 1]
y_pred_test = (proba_test >= 0.54).astype(int)

In [ ]:
#SHAP

In [ ]:
import shap
import matplotlib.pyplot as plt

# --- Explainer SHAP pour Logistic Regression ---
explainer = shap.Explainer(best_model.named_steps["clf"], best_model.named_steps["scal"].transform(X))
shap_values = explainer(best_model.named_steps["scal"].transform(X))

# --- Importance globale des variables ---
shap.summary_plot(shap_values, X, plot_type="bar")

# --- Effet détaillé (global + directionnel) ---
shap.summary_plot(shap_values, X)

# --- Analyse locale pour un client particulier ---
i = 42  # exemple de client
shap.force_plot(explainer.expected_value, shap_values[i,:], X.iloc[i,:])


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
LinearExplainer explainer: 307508it [00:10, 367.49it/s]                              


In [ ]:
API : python
Flask : web
FASTAPI : web
Recevoir un numéro de client et de faire une prédiction

In [ ]:
besoin de nouveaux outils : github  Creer un compte pour suivre les modif de la base de code
suivre tous les changements
https://openclassrooms.com/fr/paths/164/projects/632/courses

IDE : environnenmt de dev : pycham et vscode

In [ ]:
Dashboard

site web qui cree des prediction
Hebergeur : Heroku
demander un numéro a OC pour creer un Github Etudiant
PythonAnywhere
AWS, AZURE, Microsofts

In [ ]:
besoi